# Quanto tempo falta para um precatório sair?

Imagine uma firma de advocacia ou uma financeira olhando para um precatório alimentício e pensando: *vale a pena comprar esse direito agora?* O comprador paga hoje, com desconto, e só recebe quando o ente público finalmente pagar. Se isso acontecer em um ano, ótimo negócio. Se demorar dez, o dinheiro fica parado.

Este notebook tenta responder a essa pergunta com dados: **qual a chance de um precatório em aberto ser pago nos próximos 18 meses?**

Usamos os dados públicos do TJRJ, juntando os precatórios que já foram pagos com a fila dos que ainda estão esperando. O caminho é este:

1. Carregar os dados
2. Limpar e filtrar (só alimentícios, sem suspensos)
3. Montar a pergunta que o modelo vai aprender
4. Escolher as informações (features) que o modelo pode usar
5. Separar treino, validação e teste
6. Treinar quatro modelos diferentes
7. Comparar os modelos
8. Olhar de perto o vencedor
9. Gerar o ranking dos precatórios em aberto

> **Nota de revisão (outubro de 2026)**
>
> Na primeira versão deste notebook eu deixei passar dois pontos sobre como o TJRJ organiza os precatórios. Fui conferir na [página de perguntas frequentes do Portal de Precatórios](https://www.tjrj.jus.br/web/precatorios/perguntas-frequentes) e corrigi:
>
> 1. **A `ordem_cronologica` não é uma fila única.** Ela é a ordem de autuação dentro de cada ente devedor e de cada ano de orçamento, e recomeça do 1 em cada um. Conferi na base: o mesmo número se repete entre entes e entre anos, mas nunca dentro do mesmo ente e ano. Na versão anterior o número entrava cru no modelo, então uma ordem 500 podia ser o fim da fila de um município pequeno ou o começo da fila do Estado. Agora ela entra como **posição relativa** dentro da fila (0 = início, 1 = fim).
> 2. **Eu não usei o prazo legal de pagamento.** No regime comum, o ente tem que pagar até 31 de dezembro do ano do orçamento do precatório. No regime especial, o prazo para quitar o que está em atraso é 31/12/2029 (EC 109/21). Agora o modelo recebe quantos **dias faltam até esse prazo** no regime comum. No regime especial, como o prazo é igual para todos, ele só vai ajudar quando o modelo for treinado com várias datas de corte (validação temporal, próximo passo).
>
> A boa notícia é que a `ordem_cronologica` **não vazava informação do futuro**: ela é definida na autuação e não muda depois (a correlação com a data de protocolo, dentro de cada ente e ano, é praticamente 1). Os resultados da versão anterior eram válidos; esta versão só tenta extrair melhor o que a coluna e as regras de pagamento dizem.
>
> Ainda vale lembrar: a ordem cronológica não é exatamente a ordem de saída. Precatórios com prioridade constitucional (idosos, doentes graves, pessoas com deficiência) vão para uma lista prioritária separada, e a ordem real de pagamento junta as filas de todos os entes de uma mesma origem.

## 1. Carregando os dados

A base fica no Google Drive. Ela tem os precatórios já finalizados e os que ainda estão na fila, cada um com a data em que começou e, quando já terminou, a data em que foi finalizado.

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
os.makedirs('images', exist_ok=True)

id_arquivo = '1vWVNaCcEnrN7GxnLQYh4KI5VdDE8vA2m'
url = f'https://drive.google.com/uc?export=download&id={id_arquivo}'
precatorios = pd.read_csv(url)

print(f'{precatorios.shape[0]:,} precatórios e {precatorios.shape[1]} colunas')
precatorios.head()

61,718 precatórios e 27 colunas


,lista,id_entidade,entidade_devedora,entidade_pai,regime,numero_precatorio,situacao,natureza,data_protocolo,ano_orcamento,valor_historico,saldo,ordem_cronologica,ordem_pagamento,prioridade,parcelas,quitado,processo_originario,ultima_fase,data_ultima_fase,possui_cessao,possui_herdeiro,data_inicio,data_fim,finalizado,duracao_dias,data_referencia
0,pago,1,Estado do Rio de Janeiro,Estado do Rio de Janeiro,E,2014.02909-6,Arquivado,Alimentícia,2014-09-16,2016,10687.41,0,82,0,NaN,5/5,Sim,0183608-33.1999.8.19.0001,Arquivamento,2019-01-29,Não,Não,2014-09-16,2019-01-29,1,1596,2026-10-03
1,pago,86,INSS - INSTITUTO NACIONAL DO SEGURO SOCIAL,INSS - INSTITUTO NACIONAL DO SEGURO SOCIAL,O,2015.01832-2,Arquivado,Alimentícia,2015-06-02,2016,113667.41,0,286,0,NaN,5/5,Sim,0000228-23.1991.8.19.0054,Arquivamento,2019-01-29,Não,Não,2015-06-02,2019-01-29,1,1337,2026-10-03
2,pago,1,Estado do Rio de Janeiro,Estado do Rio de Janeiro,E,2016.03152-7,Arquivado,Alimentícia,2016-06-17,2017,61599.45,0,2116,0,NaN,5/5,Sim,0119271-25.2005.8.19.0001,Arquivamento,2022-10-14,Não,Não,2016-06-17,2022-10-14,1,2310,2026-10-03
3,pago,1,Estado do Rio de Janeiro,Estado do Rio de Janeiro,E,2018.03924-0,Arquivado,Alimentícia,2018-08-16,2020,445696.34,0,70,0,NaN,5/5,Sim,0103152-03.2016.8.19.0001,Juntada,2024-09-24,Sim,Não,2018-08-16,2024-09-24,1,2231,2026-10-03
4,pago,1,Estado do Rio de Janeiro,Estado do Rio de Janeiro,E,2021.08040-9,Arquivado,Alimentícia,2021-08-09,2023,46271.82,0,107,0,NaN,5/5,Sim,0052570-38.2012.8.19.0001,Juntada,2025-02-17,Não,Sim,2021-08-09,2025-02-17,1,1288,2026-10-03


## 2. Limpeza

Algumas colunas saem logo de cara. Umas porque não ajudam (como `parcelas` e `entidade_pai`), outras por um motivo mais traiçoeiro: `possui_cessao`, `possui_herdeiro`, `ultima_fase` e `prioridade` só estão preenchidas em **um** dos grupos. Se ficassem, o modelo aprenderia que “coluna preenchida = finalizado” e acertaria por um motivo que não existe na vida real.

Colunas como `situacao` e `lista` **continuam na tabela** para a gente consultar, mas não entram no modelo, porque elas praticamente entregam a resposta.

Aproveitamos também para garantir que valores e datas estejam no formato certo (parte dos valores veio com vírgula decimal).

In [ ]:
colunas_descartadas = ['id_entidade', 'entidade_pai', 'data_ultima_fase', 'parcelas', 'data_referencia',
                       'possui_cessao', 'possui_herdeiro', 'ultima_fase', 'prioridade']
precatorios = precatorios.drop(columns=colunas_descartadas, errors='ignore')

for col in ['valor_historico', 'saldo']:
    precatorios[col] = pd.to_numeric(precatorios[col].astype(str).str.replace(',', '.'), errors='coerce')

precatorios['data_protocolo'] = pd.to_datetime(precatorios['data_protocolo'])
precatorios['data_fim'] = pd.to_datetime(precatorios['data_fim'])

precatorios.columns.tolist()

['lista',
 'entidade_devedora',
 'regime',
 'numero_precatorio',
 'situacao',
 'natureza',
 'data_protocolo',
 'ano_orcamento',
 'valor_historico',
 'saldo',
 'ordem_cronologica',
 'ordem_pagamento',
 'quitado',
 'processo_originario',
 'data_inicio',
 'data_fim',
 'finalizado',
 'duracao_dias']

Agora os filtros:

- **Só alimentícios.** É o tipo que interessa (salários, aposentadorias, pensões) e também o único que tem casos finalizados na base. Os comuns estão todos em aberto, o que ensinaria ao modelo que “comum nunca sai”.
- **Sem suspensos.** Um precatório suspenso fica parado até a suspensão cair. O modelo não consegue enxergar isso nas features e acabaria dando chance alta para algo que não vai andar.
- **Sem INSS.** O INSS paga benefícios previdenciários, não salários de servidores.

In [ ]:
TIRAR_INSS = True

antes = len(precatorios)
precatorios = precatorios[precatorios['natureza'].str.contains('Aliment', na=False)]
precatorios = precatorios[~precatorios['situacao'].str.contains('Suspens', na=False)]
if TIRAR_INSS:
    precatorios = precatorios[~precatorios['entidade_devedora'].str.startswith('INSS', na=False)]

print(f'Saímos de {antes:,} para {len(precatorios):,} precatórios.')
print()
print(precatorios['situacao'].value_counts())
print()
print(precatorios['finalizado'].value_counts().rename({1: 'finalizados', 0: 'em aberto'}))

Saímos de 61,718 para 46,042 precatórios.

situacao
Ativo                          38728
Arquivado                       7302
Para arquivar                      5
Dispensa de Provisionamento        4
Impugnado                          3
Name: count, dtype: int64

finalizado
em aberto      38742
finalizados     7300
Name: count, dtype: int64


## 3. A pergunta: sai ou não sai em 18 meses?

Aqui está o pulo do gato. Quem compra um precatório paga **hoje**, então o que importa é quanto tempo **ainda falta**, não quanto tempo já passou.

Para ensinar isso ao modelo, a gente “volta no tempo”: escolhe uma data de corte 18 meses antes do último pagamento registrado e olha quais precatórios estavam em aberto naquele dia. Depois confere quais deles foram pagos nos 18 meses seguintes.

- **1** = saiu dentro da janela de 18 meses
- **0** = continuou esperando

É como se o modelo estivesse no passado tentando adivinhar o futuro, só que a gente já sabe a resposta e pode conferir.

In [ ]:
MESES = 18

ultimo_pagamento = precatorios['data_fim'].max()
data_corte = ultimo_pagamento - pd.DateOffset(months=MESES)
fim_janela = data_corte + pd.DateOffset(months=MESES)

print(f'Data de corte: {data_corte:%d/%m/%Y}')
print(f'Janela observada: {data_corte:%d/%m/%Y} até {fim_janela:%d/%m/%Y}')

estava_aberto = (precatorios['data_protocolo'] <= data_corte) & (
    (precatorios['finalizado'] == 0) | (precatorios['data_fim'] > data_corte))
treino = precatorios[estava_aberto].copy()

treino['alvo'] = ((treino['finalizado'] == 1) & (treino['data_fim'] <= fim_janela)).astype(int)
treino['idade_dias'] = (data_corte - treino['data_protocolo']).dt.days

print()
print(treino['alvo'].value_counts().rename({0: 'não saiu', 1: 'saiu em 18 meses'}))
print(f"\nSó {treino['alvo'].mean():.1%} saíram na janela.")

Data de corte: 02/04/2025
Janela observada: 02/04/2025 até 02/10/2026

alvo
não saiu            28363
saiu em 18 meses     1697
Name: count, dtype: int64

Só 5.6% saíram na janela.


Repare como os casos que saíram são minoria. Isso vai importar mais à frente: um modelo preguiçoso que dissesse “não sai” para todo mundo já acertaria mais de 90% das vezes, e mesmo assim não serviria para nada. Por isso não vamos olhar para acurácia, e sim para métricas que se importam com a classe pequena.

## 4. Features: o que o modelo pode enxergar

| feature | por que entra |
|---|---|
| `idade_dias` | há quanto tempo o precatório está na fila. Quanto mais velho, mais perto da vez dele |
| `posicao_fila` | a posição relativa na fila do seu ente e ano de orçamento (0 = início, 1 = fim). Substitui a `ordem_cronologica` crua, que recomeça do 1 em cada ente e ano |
| `dias_ate_prazo` | quantos dias faltam até o prazo legal: 31/12 do ano do orçamento (só no regime comum; o especial recebe um valor fora da escala). Negativo quando o prazo já passou |
| `log_valor` | o valor, em escala logarítmica (os valores vão de zero a mais de um bilhão) |
| `mes_protocolo` | o mês em que entrou |
| `regime_especial` | se o ente está no regime especial (atrasado, paga parcelado) ou no ordinário |
| `ent_...` | quem deve. Cada ente paga num ritmo bem diferente |

Para não explodir o número de colunas, ficam as 30 entidades com mais precatórios e o resto vira “OUTRAS”.

De fora, de propósito: `situacao`, `lista`, `quitado`, `saldo` e `ordem_pagamento` (entregam a resposta), e os anos de protocolo e orçamento como números soltos (a base de pagos só tem pagamentos recentes, então o modelo aprenderia que “antigo nunca paga”, o que é um defeito da base, não da realidade).

**Atenção:** no regime comum, `dias_ate_prazo` é calculado a partir do ano do orçamento, então traz um pouco dessa informação de volta. Vale comparar os resultados com e sem ela.

In [ ]:
TOP_ENTIDADES = 30
principais = treino['entidade_devedora'].value_counts().head(TOP_ENTIDADES).index

# A ordem cronológica recomeça em cada ente e ano de orçamento.
# O tamanho de cada fila é o maior número de ordem que aparece nela.
fila = ['entidade_devedora', 'ano_orcamento']
tamanho_fila = precatorios.groupby(fila)['ordem_cronologica'].max().rename('tamanho_fila')

# Prazo legal: regime comum paga até 31/12 do ano do orçamento;
# regime especial tem até 31/12/2029 (EC 109/21).
PRAZO_REGIME_ESPECIAL = pd.Timestamp('2029-12-31')
SEM_PRAZO = 99_999


def prazo_legal(df):
    prazo_comum = pd.to_datetime(df['ano_orcamento'].astype(str) + '-12-31')
    return prazo_comum.where(df['regime'] != 'E', PRAZO_REGIME_ESPECIAL)


def montar_features(df, data_ref):
    X = pd.DataFrame(index=df.index)
    X['idade_dias'] = df['idade_dias']

    tamanho = df[fila].join(tamanho_fila, on=fila)['tamanho_fila']
    X['posicao_fila'] = df['ordem_cronologica'] / tamanho

    # No regime especial o prazo é o mesmo para todos (31/12/2029). Com uma única data de corte,
    # ele vira um valor constante no treino e um valor nunca visto no ranking de hoje.
    # Por isso, por enquanto, ele só entra para o regime comum; o especial recebe um valor fora da escala.
    dias = (prazo_legal(df) - data_ref).dt.days
    X['dias_ate_prazo'] = dias.where(df['regime'] != 'E', SEM_PRAZO)
    X['log_valor'] = np.log1p(df['valor_historico'])
    X['mes_protocolo'] = df['data_protocolo'].dt.month
    X['regime_especial'] = (df['regime'] == 'E').astype(int)

    entidade = df['entidade_devedora'].where(df['entidade_devedora'].isin(principais), 'OUTRAS')
    X = pd.concat([X, pd.get_dummies(entidade, prefix='ent', dtype=int)], axis=1)

    # XGBoost e LightGBM não gostam de acentos e símbolos no nome das colunas
    X.columns = [re.sub(r'[^0-9a-zA-Z_]+', '_', c) for c in X.columns]
    return X.fillna(0)


X = montar_features(treino, data_corte)
y = treino['alvo']

# os precatórios em aberto hoje, que são os candidatos à compra
hoje = pd.Timestamp.today().normalize()
em_aberto = precatorios[precatorios['finalizado'] == 0].copy()
em_aberto['idade_dias'] = (hoje - em_aberto['data_protocolo']).dt.days
X_aberto = montar_features(em_aberto, hoje).reindex(columns=X.columns, fill_value=0)

print(f'{X.shape[1]} features para {X.shape[0]:,} precatórios')
X.head()

## 5. Treino, validação e teste

Dividimos em três pedaços:

- **Treino (60%)**: onde os modelos aprendem.
- **Validação (20%)**: onde escolhemos o corte de probabilidade de cada modelo.
- **Teste (20%)**: guardado até o fim, para uma nota honesta. Nenhuma decisão é tomada olhando para ele.

O `stratify` garante que a proporção de “saiu” seja a mesma nos três.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_temp, y_tr, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f'Treino: {len(X_tr):,} | Validação: {len(X_val):,} | Teste: {len(X_te):,}')

## 6. Os modelos

Quatro candidatos, do mais simples ao mais sofisticado:

- **Regressão Logística**: a referência. Simples e fácil de explicar. Se os outros não baterem ela, algo está errado.
- **Random Forest**: várias árvores de decisão votando juntas.
- **XGBoost**: árvores construídas em sequência, cada uma corrigindo os erros da anterior. Costuma ir muito bem em dados tabulares.
- **LightGBM**: a mesma ideia do XGBoost, mais rápido.

Como os casos que saíram são poucos, todos recebem um “peso extra” para essa classe (`class_weight` ou `scale_pos_weight`). Sem isso, eles tenderiam a ignorá-la.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

peso = (y_tr == 0).sum() / (y_tr == 1).sum()   # quantos 'não saiu' para cada 'saiu'

modelos = {
    'Regressão Logística': make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=2000)),
    'Random Forest': RandomForestClassifier(
        n_estimators=400, min_samples_leaf=5, class_weight='balanced',
        random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8,
        colsample_bytree=0.8, scale_pos_weight=peso, eval_metric='aucpr',
        random_state=42, n_jobs=-1),
    'LightGBM': LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=31, subsample=0.8,
        subsample_freq=1, colsample_bytree=0.8, scale_pos_weight=peso,
        random_state=42, verbose=-1),
}

for nome, modelo in modelos.items():
    modelo.fit(X_tr, y_tr)
    print(f'{nome}: pronto')

## 7. Quem foi melhor?

As métricas que importam aqui:

- **AUC**: o quanto o modelo consegue colocar quem sai na frente de quem não sai. 0,5 é chute, 1,0 é perfeito.
- **PR-AUC**: parecida, mas focada na classe pequena. É a mais honesta quando os positivos são raros, e é por ela que vamos ordenar.
- **Precision**: dos precatórios que o modelo apontou como “vai sair”, quantos saíram mesmo. Para quem compra, é a mais importante: cada erro aqui é dinheiro parado.
- **Recall**: dos que saíram, quantos o modelo encontrou. Recall baixo significa deixar passar oportunidades.
- **F1**: o equilíbrio entre as duas.

O corte de cada modelo (a partir de qual % a gente diz “vai sair”) é escolhido na validação, buscando o melhor F1. Só depois medimos no teste.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             precision_score, recall_score, f1_score)


def melhor_corte(y_true, proba):
    prec, rec, cortes = precision_recall_curve(y_true, proba)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    return cortes[np.argmax(f1)]


resultados = []
cortes = {}
for nome, modelo in modelos.items():
    cortes[nome] = melhor_corte(y_val, modelo.predict_proba(X_val)[:, 1])
    proba_te = modelo.predict_proba(X_te)[:, 1]
    pred_te = (proba_te >= cortes[nome]).astype(int)
    resultados.append({
        'modelo': nome,
        'AUC': roc_auc_score(y_te, proba_te),
        'PR-AUC': average_precision_score(y_te, proba_te),
        'corte': cortes[nome],
        'precision': precision_score(y_te, pred_te),
        'recall': recall_score(y_te, pred_te),
        'F1': f1_score(y_te, pred_te),
    })

resultados = pd.DataFrame(resultados).set_index('modelo').sort_values('PR-AUC', ascending=False)
resultados.round(3)

O gráfico abaixo mostra a troca entre precision e recall para cada modelo. Quanto mais a curva fica no canto superior direito, melhor. A linha tracejada é o que um chute conseguiria.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for nome, modelo in modelos.items():
    prec, rec, _ = precision_recall_curve(y_te, modelo.predict_proba(X_te)[:, 1])
    pr_auc = resultados.loc[nome, 'PR-AUC']
    ax.plot(rec, prec, label=f'{nome} (PR-AUC {pr_auc:.2f})')

ax.axhline(y_te.mean(), color='gray', ls='--', label='chute')
ax.set_xlabel('Recall: dos que saíram, quantos o modelo achou')
ax.set_ylabel('Precision: dos apontados, quantos saíram')
ax.set_title('Precision x Recall no conjunto de teste')
ax.legend()
plt.savefig('images/curva_precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. O vencedor, de perto

Pegamos o modelo com maior PR-AUC e olhamos a matriz de confusão dele.

Se a precision estiver baixa demais para o seu gosto, dá para ser mais exigente: coloque um valor em `CORTE_MANUAL` (por exemplo `0.7`). O modelo vai apontar menos precatórios, mas com mais chance de acertar.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report

CORTE_MANUAL = None   # ex.: 0.7 para priorizar precision

melhor_nome = resultados.index[0]
melhor_modelo = modelos[melhor_nome]
corte_final = cortes[melhor_nome] if CORTE_MANUAL is None else CORTE_MANUAL

proba_te = melhor_modelo.predict_proba(X_te)[:, 1]
pred_te = (proba_te >= corte_final).astype(int)

print(f'Melhor modelo: {melhor_nome}, corte de {corte_final:.0%}')
print()
print(classification_report(y_te, pred_te, target_names=['Não saiu', 'Saiu em 18 meses']))

ConfusionMatrixDisplay(confusion_matrix(y_te, pred_te),
                       display_labels=['Não saiu', 'Saiu em 18 meses']).plot(cmap='Blues', values_format='d')
plt.title(f'{melhor_nome}: matriz de confusão')
plt.savefig('images/matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

Como ler a matriz:

| | previu “não saiu” | previu “saiu” |
|---|---|---|
| **não saiu de verdade** | acerto | alarme falso (dinheiro parado) |
| **saiu de verdade** | oportunidade perdida | acerto |

E o que mais pesou na decisão do modelo:

In [ ]:
if hasattr(melhor_modelo, 'feature_importances_'):
    importancia = pd.Series(melhor_modelo.feature_importances_, index=X.columns)
else:
    importancia = pd.Series(np.abs(melhor_modelo[-1].coef_[0]), index=X.columns)

importancia.sort_values().tail(15).plot.barh(figsize=(8, 6), title=f'O que mais pesa no {melhor_nome}')
plt.savefig('images/importancia_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. O ranking

Agora a parte que interessa para quem vai comprar: aplicamos o modelo vencedor em todos os precatórios que estão em aberto hoje e ordenamos pela chance de saírem nos próximos 18 meses.

In [ ]:
em_aberto['chance_18_meses_%'] = (melhor_modelo.predict_proba(X_aberto)[:, 1] * 100).round(1)

ranking = (em_aberto[['numero_precatorio', 'entidade_devedora', 'situacao', 'data_protocolo',
                      'valor_historico', 'saldo', 'chance_18_meses_%']]
           .sort_values('chance_18_meses_%', ascending=False))

acima_do_corte = (ranking['chance_18_meses_%'] >= corte_final * 100).sum()
print(f'{acima_do_corte:,} de {len(ranking):,} precatórios em aberto passam do corte de {corte_final:.0%}')
ranking.head(20)

In [ ]:
ranking.to_csv('ranking_precatorios_18_meses.csv', index=False)

# empacota o ranking e os gráficos para baixar e colocar no repositório
import shutil
shutil.make_archive('resultados', 'zip', '.', 'images')

from google.colab import files
files.download('ranking_precatorios_18_meses.csv')
files.download('resultados.zip')

## Antes de usar o resultado

Alguns cuidados para não ler mais do que o modelo consegue dizer:

- **A porcentagem serve para ordenar, não é garantia.** Um precatório com 95% é um candidato melhor que um com 60%, mas 95% não quer dizer “certeza”. Modelos de árvore costumam exagerar nos extremos.
- **A base de pagos só cobre pagamentos recentes.** Precatórios pagos há muitos anos não aparecem nela. Por isso os anos ficaram fora do modelo.
- **A data de fim é aproximada.** Ela vem da última movimentação registrada, que às vezes é um pouco depois do pagamento em si.
- **O mundo muda.** Se um ente entrar em crise ou receber um aporte, o ritmo de pagamento muda junto.